# Распределение товаров по торговым точкам через роевой интеллект

## Библиотеки

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor  # для многопоточности
from tqdm import tqdm  # чтобы видеть процессы

## Эвристики

Влияние продаж

In [2]:
sales_factor = 0.001

Влияние загруженности

In [3]:
load_factor = 0.0005

Сколько итераций считать

In [4]:
iterations_count = 100

Ранняя остановка. Если лучшее решение не менялось на протяжении указанного кол-ва итераций, то останваливаем алгоритм. Значение 0 значит без ранней остановки

In [5]:
early_stopping = 5

Константа выделения феромонов

In [6]:
pheromone_quantity = 1

Испарение феромонов

In [7]:
pheromone_loss = 0.5

Влияние феромонов

In [8]:
pheromone_factor = 1.8

Влияние эвристик

In [9]:
heuristic_factor = 2

Число муравьёв

In [10]:
ant_count = 16

## Подгрузка датасета

In [32]:
col_vol = "Объём товара в закрывающемся магазине (V)"
col_maxcap = "Максимальная вместимость товара в магазине (M)"
col_item = "Код товара"
col_store = "Магазин"
col_remain = "Остаток товара в магазине (R)"
col_sales = "Продажи товара в магазине (S)"
col_dist = "Распределение муравьёв"

In [33]:
items = pd.read_csv(r"./filtered_datasets/items_closing_store.csv", sep=";")
items = items.set_index([col_item, col_store])
items

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239388     95                                            26.0   
239397     38                                            12.0   
239398     38                                            13.0   
239515     38                                            42.0   
239516     38                                            42.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239388     95                            1.500000   
239397     38                            1.116279   
239398     38                            2.023256   
239515     38                            1.796296   
239516     38                            1.740741   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                    8   
           56                                                    8   
           57                                                    8   
           60                                                    8   
           61                                                    8   
...                                                            ...   
239388     95                                                   25   
239397     38                                                   24   
239398     38                                                   24   
239515     38                                                   30   
239516     38                                                   24   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239388     95                                20.0  
239397     38                                21.0  
239398     38                                45.0  
239515     38                                35.0  
239516     38                                28.0  

[413689 rows x 4 columns]

Со всем датасетом работать очень медленно. Гораздо лучше будет сделать несколько матриц

$i$ - товар, $i ∈ [1, N]$, $N$ - кол-во товаров

$j$ - магазин, $j ∈ [1, M]$, $M$ - кол-во магазинов

*closing_volume* - вектор размерности $N×1$, показывает кол-во товара $i$ в закрывающемся магазине

*sales* - матрица размерности $N×M$, показывает продажи товара $i$ в магазине $j$

*maxcap* - матрица размерности $N×M$, показывает вместимость на полках товара $i$ в магазине $j$

*remains* - матрица размерности $N×M$, показывает остатки товара $i$ в магазине $j$

*pheromones* - матрица размерности $N×M$, показывает феромоны для распределения товара $i$ в магазин $j$

Если товар $i$ не числится в магазине $j$, то тогда соответственные значения $(i, j)$ во всех матрицах равны 0

In [13]:
items_id = items.index.get_level_values(0).unique().values
N = items_id.shape[0]
N

5366

In [14]:
stores_id = items.index.get_level_values(1).unique().values
M = stores_id.shape[0]
M

98

Переиндексация для создания пар $(i, j)$

In [15]:
full_index = pd.MultiIndex.from_product([items_id, stores_id], names=[col_item, col_store])
df_full = items.reindex(full_index)
df_full

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239221     88                                             NaN   
           105                                            NaN   
           29                                             NaN   
           27                                             NaN   
           113                                           17.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239221     88                                 NaN   
           105                                NaN   
           29                                 NaN   
           27                                 NaN   
           113                           0.302326   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                  8.0   
           56                                                  8.0   
           57                                                  8.0   
           60                                                  8.0   
           61                                                  8.0   
...                                                            ...   
239221     88                                                  NaN   
           105                                                 NaN   
           29                                                  NaN   
           27                                                  NaN   
           113                                                10.0   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239221     88                                 NaN  
           105                                NaN  
           29                                 NaN  
           27                                 NaN  
           113                                5.0  

[525868 rows x 4 columns]

Превращаем всё теперь в матрицы

In [16]:
closing_volume = items.groupby(level=0).first()[col_vol].values
sales = df_full[col_sales].values.reshape(N, M)
maxcap = df_full[col_maxcap].values.reshape(N, M)
remains = df_full[col_remain].values.reshape(N, M)

Заполняем пропуски нулями

In [17]:
sales = np.nan_to_num(sales, nan=0)
maxcap = np.nan_to_num(maxcap, nan=0)
remains = np.nan_to_num(remains, nan=0)

## Муравьиный алгоритм

### Логика движения муравья

Сначала фиксируется товар, потом для него ищется распределение по магазинам, опираясь на случайный выбор по вероятности

In [18]:
def ant_distribution(closing_volume, sales, maxcap, remains, pheromones):    
    v, s, m, r, w = closing_volume, sales, maxcap, remains, pheromones
    c = np.zeros((N, M), dtype=np.int64)  # изначально распределение равно 0
    
    for i in range(N):        
        item_count = v[i]
        zero_sales = np.max(s[i]) == 0.0  # если у товара везде нулевые продажи, то веса считаются по-другому

        load_weights = np.divide(m[i], (m[i] + r[i]), out=np.zeros(M, dtype=np.float64), where=m[i] > 0)
        weights = w[i] ** pheromone_factor

        if not zero_sales:
            weights *= (s[i] * r[i] / (1 + r[i]) * load_weights) ** heuristic_factor

        else:  # если продажи везде нулевые, то веса считаем только через загруженность
            weights *= load_weights ** heuristic_factor
            
        while item_count > 0:  # пока не кончатся товары для распределения
            cumulative = np.cumsum(weights)  # кумулятивная сумма весов, получится отрезок, равный сумме весов
            selected = np.random.rand() * cumulative[-1]  # на этом отрезке случайно выберается промежуток
            j = np.searchsorted(cumulative, selected, side="right")  # и потом ищется индекс, где этот промежуток можно вместить, т.е. выбирается магазин 
            j = np.min([j, M-1])  # чтобы случайно не вылететь за границу

            if m[i][j] == 0.0:  # если товар в этом магазине не введён в продажу, то ищем другой
                continue
            
            c[i][j] += 1  # распределили 1 штуку товара

            if not zero_sales:
                weights[j] = w[i][j] ** pheromone_factor * (s[i][j] * (c[i][j] + r[i][j]) / (1 + r[i][j]) * m[i][j] / (m[i][j] + c[i][j] + r[i][j])) ** heuristic_factor

            else:
                weights[j] = w[i][j] ** pheromone_factor * (m[i][j] / (m[i][j] + c[i][j] + r[i][j])) ** heuristic_factor

            item_count -= 1

    sales_rating = sales_factor * s / (1 + r)
    load_rating = np.divide(load_factor, m, out=np.zeros((N, M), dtype=np.float64), where=m > 0)  # чтобы не поделить на 0
    total_rating = np.sum((c + r) * (sales_rating - load_rating))  # оценка решения
    
    update = pheromone_quantity * np.max([total_rating, 0.0]) * (c > 0) # обновление феромонов, если оценка очень плохая и отрицательная, то не обновляем тогда

    return update, total_rating, c

### Распараллеливание

Чтобы было быстрее считать, алгоритм распараллеливает муравьёв, каждый считает своё решение, а затем всё обновляется

In [19]:
def ant_colony_optimization(closing_volume, sales, maxcap, remains):
    pheromones = np.ones((N, M), dtype=np.float64)  # изначально феромоны равны 1

    best_rating, best_soluion = None, None
    no_improvements = 0

    for i in range(iterations_count):
        print(f"\nIteration {i+1} out of {iterations_count}")
        pheromone_updates, ratings, solutions = [], [], []

        with ThreadPoolExecutor() as executor:
            futures = [executor.submit(ant_distribution, closing_volume, sales, maxcap, remains, pheromones) for _ in range(ant_count)]
    
            for future in tqdm(futures):
                result = future.result()
                pheromone_updates.append(result[0])
                ratings.append(result[1])
                solutions.append(result[2])

        pheromones = (1 - pheromone_loss) * pheromones + np.sum(pheromone_updates, axis=0)  # глобальное обновление феромонов
        max_index = np.argmax(ratings)

        current_best_rating = ratings[max_index]
        current_best_solution = solutions[max_index]
        no_improvements += 1

        print("Best rating for the current iteration:", current_best_rating)

        if best_rating is None or best_solution is None or (current_best_rating > best_rating):
            best_rating = current_best_rating
            best_solution = current_best_solution
            no_improvements = 0

        print("Total best rating:", best_rating)

        if early_stopping > 0 and no_improvements >= early_stopping:
            print("Early stopping")
            break
            
    return best_rating, best_solution        

In [20]:
best_rating, best_solution = ant_colony_optimization(closing_volume, sales, maxcap, remains)


Iteration 1 out of 100


100%|██████████| 16/16 [00:22<00:00,  1.41s/it]


Best rating for the current iteration: 49.21757947222177
Total best rating: 49.21757947222177

Iteration 2 out of 100


100%|██████████| 16/16 [00:30<00:00,  1.88s/it]


Best rating for the current iteration: 54.812924687948694
Total best rating: 54.812924687948694

Iteration 3 out of 100


...

Best rating for the current iteration: 59.09704517836889
Total best rating: 59.213902521392114

Iteration 26 out of 100


100%|██████████| 16/16 [00:28<00:00,  1.77s/it]


Best rating for the current iteration: 59.10633498069079
Total best rating: 59.213902521392114

Iteration 27 out of 100


100%|██████████| 16/16 [00:29<00:00,  1.85s/it]

Best rating for the current iteration: 59.086695047345756
Total best rating: 59.213902521392114
Early stopping


In [21]:
best_rating

np.float64(59.213902521392114)

In [22]:
best_solution

array([[4, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5366, 98))

## Выгрузка решения

In [23]:
def output_csv(df, filename, with_index=False):
    df_dirpath = Path(r"./solutions")
    df_dirpath.mkdir(parents=True, exist_ok=True)

    df_filepath = df_dirpath / filename

    with df_filepath.open("w") as f:
        df.to_csv(f, sep=";", lineterminator="\n", index=with_index)

In [34]:
df_solution = pd.DataFrame(best_solution, index=items_id, columns=stores_id).stack()
items[col_dist] = df_solution.reindex(items.index).values
items

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239388     95                                            26.0   
239397     38                                            12.0   
239398     38                                            13.0   
239515     38                                            42.0   
239516     38                                            42.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239388     95                            1.500000   
239397     38                            1.116279   
239398     38                            2.023256   
239515     38                            1.796296   
239516     38                            1.740741   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                    8   
           56                                                    8   
           57                                                    8   
           60                                                    8   
           61                                                    8   
...                                                            ...   
239388     95                                                   25   
239397     38                                                   24   
239398     38                                                   24   
239515     38                                                   30   
239516     38                                                   24   

                    Остаток товара в магазине (R)  Распределение муравьёв  
Код товара Магазин                                                         
226249     54                                16.0                       4  
           56                                22.0                       0  
           57                                19.0                       0  
           60                                28.0                       0  
           61                                22.0                       0  
...                                           ...                     ...  
239388     95                                20.0                       0  
239397     38                                21.0                       0  
239398     38                                45.0                       9  
239515     38                                35.0                       0  
239516     38                                28.0                       3  

[413689 rows x 5 columns]

In [35]:
output_csv(items, "ant_solution.csv", with_index=True)